# Assignment:
Since, we i got an assignment of creating a dataset from the website called "TMDB", a movies based website and i have to extract name of the movie, description of the movie, and it gernre. we have given an API key as well.

In [12]:
# Use existing API_KEY if valid; otherwise fallback to key found in genre_url
# tsting API connection and response structure
if API_KEY == "YOUR_API_KEY" and "api_key=" in genre_url:
	API_KEY = genre_url.split("api_key=")[1].split("&")[0]

url = f"https://api.themoviedb.org/3/movie/top_rated?api_key={API_KEY}&page=1"

try:
	res = session.get(url, timeout=30)
except requests.exceptions.ConnectionError:
	# one immediate retry for transient network reset
	res = session.get(url, timeout=30)

res.raise_for_status()
print(res.status_code)
print(res.json().keys())

200
dict_keys(['page', 'results', 'total_pages', 'total_results'])


In [15]:
# Get genres list to map genre ids to names

if API_KEY == "YOUR_API_KEY" and "api_key=" in genre_url:
    API_KEY = genre_url.split("api_key=")[1].split("&")[0]

url = f"https://api.themoviedb.org/3/genre/movie/list?api_key={API_KEY}"
res = session.get(url, timeout=30)
res.raise_for_status()

data = res.json()

genre_dict = {}

for g in data.get("genres", []):
    genre_dict[g["id"]] = g["name"]

if genre_dict:
    print(genre_dict)
else:
    print(data.get("status_message", "No genres found"))

{28: 'Action', 12: 'Adventure', 16: 'Animation', 35: 'Comedy', 80: 'Crime', 99: 'Documentary', 18: 'Drama', 10751: 'Family', 14: 'Fantasy', 36: 'History', 27: 'Horror', 10402: 'Music', 9648: 'Mystery', 10749: 'Romance', 878: 'Science Fiction', 10770: 'TV Movie', 53: 'Thriller', 10752: 'War', 37: 'Western'}


In [17]:
# Use existing API_KEY if valid; otherwise try to extract it from known URLs
if API_KEY == "YOUR_API_KEY":
    for candidate in [genre_url, url]:
        if "api_key=" in candidate:
            extracted_key = candidate.split("api_key=")[1].split("&")[0]
            if extracted_key and extracted_key != "YOUR_API_KEY":
                API_KEY = extracted_key
                break

def get_json_with_retry(request_url, timeout=30):
    try:
        r = session.get(request_url, timeout=timeout)
    except requests.exceptions.ConnectionError:
        # one immediate retry for transient reset errors
        r = session.get(request_url, timeout=timeout)
    r.raise_for_status()
    return r.json()

# get genres
genre_url = f"https://api.themoviedb.org/3/genre/movie/list?api_key={API_KEY}"
genre_data = get_json_with_retry(genre_url)
genre_dict = {g["id"]: g["name"] for g in genre_data.get("genres", [])}

# get movies (page 1 only)
url = f"https://api.themoviedb.org/3/movie/top_rated?api_key={API_KEY}&page=1"
data = get_json_with_retry(url)

movies = []

for m in data.get("results", []):
    genres = []
    for gid in m.get("genre_ids", []):
        genres.append(genre_dict.get(gid, "Unknown"))

    movie = {
        "name": m.get("title", ""),
        "description": m.get("overview", ""),
        "genre": ", ".join(genres)
    }

    movies.append(movie)

for m in movies[:3]:
    print(m)

{'name': 'The Shawshank Redemption', 'description': 'Imprisoned in the 1940s for the double murder of his wife and her lover, upstanding banker Andy Dufresne begins a new life at the Shawshank prison, where he puts his accounting skills to work for an amoral warden. During his long stretch in prison, Dufresne comes to be admired by the other inmates -- including an older prisoner named Red -- for his integrity and unquenchable sense of hope.', 'genre': 'Drama, Crime'}
{'name': 'The Godfather', 'description': 'Spanning the years 1945 to 1955, a chronicle of the fictional Italian-American Corleone crime family. When organized crime family patriarch, Vito Corleone barely survives an attempt on his life, his youngest son, Michael steps in to take care of the would-be killers, launching a campaign of bloody revenge.', 'genre': 'Drama, Crime'}
{'name': 'The Godfather Part II', 'description': 'In the continuing saga of the Corleone crime family, a young Vito Corleone grows up in Sicily and in

In [19]:
# Reuse API key from previous cells; if placeholder, extract from known URLs
if API_KEY == "YOUR_API_KEY":
    for candidate_url in [genre_url, url, candidate]:
        if isinstance(candidate_url, str) and "api_key=" in candidate_url:
            extracted = candidate_url.split("api_key=")[1].split("&")[0]
            if extracted and extracted != "YOUR_API_KEY":
                API_KEY = extracted
                break

# genres
genre_data = session.get(
    f"https://api.themoviedb.org/3/genre/movie/list?api_key={API_KEY}",
    timeout=30
).json()

genres_list = genre_data.get("genres", [])
if not genres_list:
    raise ValueError(genre_data.get("status_message", "Could not load genres. Check API key."))

genre_dict = {g["id"]: g["name"] for g in genres_list}

movies = []

# loop pages
for page in range(1, 472):
    page_url = f"https://api.themoviedb.org/3/movie/top_rated?api_key={API_KEY}&page={page}"
    data = session.get(page_url, timeout=30).json()

    for m in data.get("results", []):
        genres = [genre_dict.get(gid, "Unknown") for gid in m.get("genre_ids", [])]

        movies.append({
            "name": m.get("title", ""),
            "description": m.get("overview", ""),
            "genre": ", ".join(genres)
        })

print("Total movies:", len(movies))

Total movies: 9420


In [ ]:
import pandas as pd

df = pd.DataFrame(movies)
df.to_csv("TMDB dataset.csv", index=False)

print("CSV saved!")

CSV saved!
